# Constitutive Model Calibration (PMDCo / OBI): from data entry to RDF

This notebook shows how to record a constitutive model calibration and convert
that record into a standardised, machine-readable RDF graph.

**You only need to edit one file:** `docs/example.input.json`. Everything
else is automatic.

This schema **extends** the generic `simulation/step/PMDCo/` schema with two
additional concepts:

- the **model type** (e.g. Hockett-Sherby, Swift, Voce, Hollomon, Johnson-Cook)
- the **calibrated parameters** output by the fitting algorithm
  (e.g. `sigma_sat = 780 MPa`, `n = 0.68`)

A calibration step captures:

- a **name** and optional **description**
- the **input experimental datasets** (e.g. tensile test results)
- the **output material card** produced
- which **characterisation step** preceded this calibration
- optional **algorithmic settings** (optimiser, strain range, iterations)
- the **fitted parameter values** with units


## Inheritance

This schema is a specialisation of `simulation/step/PMDCo/`. It reuses all
base-schema fields (`has_specified_input`, `has_specified_output`,
`preceded_by`, `has_process_condition`) and adds `model_type` and
`calibrated_parameters`. See
[docs/schema-patterns.md](../../../docs/schema-patterns.md) for details on how
inheritance works in this repository.


## What the notebook does

```
example.input.json
  |  model type, inputs, outputs, algorithm settings, and fitted parameters
  |
  v
Transform
  |  converts your plain JSON into a structured OO-LD document
  |
  v
RDF graph
  |  machine-readable, ontology-aligned triples
  |
  v
SHACL validation  ->  confirms the graph is structurally correct
SPARQL inspect    ->  shows the key connections and parameter values
```


## Environment setup

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install jupyterlab
jupyter lab
```

In [ ]:
%pip install -q jsonata-python rdflib pyshacl pyyaml

In [ ]:
import sys, json, pathlib, yaml, pyshacl, rdflib
from jsonata.jsonata import Jsonata
from importlib.metadata import version

print("Python        :", sys.executable)
print("jsonata-python", version("jsonata-python"))
print("rdflib        ", version("rdflib"))
print("pyshacl       ", version("pyshacl"))

HERE   = pathlib.Path().resolve()   # docs/
SCHEMA = HERE.parent                # simulation/model-calibration/PMDCo/
BASE   = SCHEMA.parent.parent / "step" / "PMDCo"  # simulation/step/PMDCo/

## Step 1: Describe your calibration

Edit `docs/example.input.json` with your data, then run this cell to load it.

| Field | Required | Description |
|---|---|---|
| `step_name` | yes | A name for this calibration step |
| `model_type` | yes | Flow curve model (Hockett-Sherby, Swift, Voce, Hollomon, Johnson-Cook) |
| `description` | no | Free-text explanation |
| `inputs` | no | IRIs of experimental datasets used for fitting |
| `outputs` | no | IRIs of the material card(s) produced |
| `preceded_by` | no | IRIs of characterisation steps that provide the data |
| `conditions` | no | Algorithmic settings (optimiser, strain range, iterations, ...) |
| `calibrated_parameters` | no | Fitted parameter values, each with `name`, `value`, and optional `unit` |
| `step_id` | no | Custom IRI slug; auto-derived from `step_name` if omitted |

**Supported model types and their parameters:**

| Model | Parameters |
|---|---|
| Hockett-Sherby | `sigma_sat`, `sigma_0`, `c`, `n` |
| Swift | `C`, `epsilon_0`, `n` |
| Voce | `sigma_sat`, `sigma_0`, `c` |
| Hollomon | `K`, `n` |
| Johnson-Cook | `A`, `B`, `n`, `C`, `m` |

In [ ]:
simplified = json.loads((HERE / "example.input.json").read_text())

print(json.dumps(simplified, indent=2))

## Step 2: Convert to OO-LD

This step transforms your plain JSON into a structured
[OO-LD](https://github.com/OO-LD/oold-python) document.
The transform assigns stable node IDs, maps field names to ontology terms,
and embeds the calibrated parameters as sub-nodes inside the calibration record.

You can also run the transform from the command line:
```
python -m jsonata -e ../simplified/transform.jsonata -i example.input.json
```

In [ ]:
expr     = (SCHEMA / "simplified" / "transform.jsonata").read_text()
oold_doc = Jsonata(expr).evaluate(simplified)

print(json.dumps(oold_doc, indent=2))

## Step 3: Convert to RDF

The OO-LD document is parsed into an RDF graph using the ontology context from
`specs/schema.oold.yaml`.

Because this schema extends `simulation/step/PMDCo/`, the `@context` from
the base schema is included in this schema's YAML file. You do not need to
load the base context separately.

In [ ]:
context = yaml.safe_load((SCHEMA / "specs" / "schema.oold.yaml").read_text())["@context"]

g = rdflib.Dataset()
g.parse(data=json.dumps({"@context": context, **oold_doc}), format="json-ld")

print(f"Graph contains {len(g)} triples.\n")
print(g.serialize(format="turtle"))

In [ ]:
# Optional: save to file
out_ttl = HERE / "output_model_calibration.ttl"
out_ttl.write_text(g.serialize(format="turtle"))
print(f"Written to {out_ttl}")

## Step 4: Validate against the SHACL shapes

Validation uses **two** shape files:

1. `simulation/step/PMDCo/specs/shape.ttl` (base shape)
2. `simulation/model-calibration/PMDCo/specs/shape.ttl` (extending shape)

Both must be loaded together because the extending shape only defines the
additional constraints (`model_type`, `calibrated_parameters`). The base
shape covers the shared fields (`label`, `has_specified_input`, etc.).
See [docs/schema-patterns.md](../../../docs/schema-patterns.md) for details on
how SHACL inheritance works.

In [ ]:
shapes_graph = rdflib.Graph()
shapes_graph.parse(str(BASE   / "specs" / "shape.ttl"))   # base shape
shapes_graph.parse(str(SCHEMA / "specs" / "shape.ttl"))   # extending shape

conforms, results_graph, _ = pyshacl.validate(
    g,
    shacl_graph = shapes_graph,
)

print(f"Conforms: {conforms}")
if not conforms:
    SH = rdflib.Namespace("http://www.w3.org/ns/shacl#")
    for result in results_graph.subjects(rdflib.RDF.type, SH.ValidationResult):
        msg  = results_graph.value(result, SH.resultMessage)
        path = results_graph.value(result, SH.resultPath)
        loc  = str(path).rsplit("#", 1)[-1].rsplit("/", 1)[-1] if path else None
        print(f"\n  x {msg}" + (f"\n    property: {loc}" if loc else ""))

## Step 5: Inspect the graph

The cells below use SPARQL to confirm that the calibration record was stored
correctly, including the fitted parameter values.

You do not need to understand SPARQL to use this notebook. Just run the cells
and check that the output matches what you entered.

In [ ]:
# Flatten the Dataset into a plain Graph for SPARQL queries
flat = rdflib.Graph()
for s, p, o, _ in g.quads():
    flat.add((s, p, o))

OBI  = rdflib.Namespace("http://purl.obolibrary.org/obo/OBI_")
SKOS = rdflib.Namespace("http://www.w3.org/2004/02/skos/core#")

# Print step label and model type
step_iri   = next(flat.subjects(rdflib.RDF.type, OBI["0000471"]))
label      = flat.value(step_iri, rdflib.RDFS.label)
model_type = flat.value(step_iri, SKOS.notation)  # model_type maps to skos:notation
print(f"Calibration step : {step_iri}")
print(f"Label            : {label}")
print(f"Model type       : {model_type}")

In [ ]:
SPARQL_IO = """
PREFIX obi:  <http://purl.obolibrary.org/obo/OBI_>
PREFIX bfo:  <http://purl.obolibrary.org/obo/BFO_>

SELECT ?role ?iri
WHERE {
  { ?step a obi:0000471 ; obi:0000293 ?iri . BIND("input"       AS ?role) }
  UNION
  { ?step a obi:0000471 ; obi:0000299 ?iri . BIND("output"      AS ?role) }
  UNION
  { ?step a obi:0000471 ; bfo:0000062 ?iri . BIND("preceded by" AS ?role) }
}
ORDER BY ?role ?iri
"""

rows = list(flat.query(SPARQL_IO))
if rows:
    print(f"{'Role':<14}  IRI")
    print("-" * 70)
    for r in rows:
        print(f"{str(r.role):<14}  {r.iri}")
else:
    print("No inputs, outputs, or predecessors recorded.")

In [ ]:
SPARQL_PARAMS = """
PREFIX obi:   <http://purl.obolibrary.org/obo/OBI_>
PREFIX iao:   <http://purl.obolibrary.org/obo/IAO_>
PREFIX qudt:  <http://qudt.org/schema/qudt/>
PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?name ?value ?unit
WHERE {
  ?step a obi:0000471 ;
        obi:0000299 ?param .
  ?param a iao:0000100 ;
         rdfs:label ?name ;
         qudt:value ?value .
  OPTIONAL { ?param qudt:hasUnit ?unit . }
}
ORDER BY ?name
"""

rows = list(flat.query(SPARQL_PARAMS))
if rows:
    print(f"{'Parameter':<20}  {'Value':>12}  Unit")
    print("-" * 50)
    for r in rows:
        val  = f"{float(r.value):>12.4f}"
        unit = str(r.unit) if r.unit else ""
        print(f"{str(r.name):<20}  {val}  {unit}")
else:
    print("No calibrated parameters recorded.")

## Summary

| Step | What happens |
|---|---|
| 1 | You fill in `example.input.json` with the model type, datasets, algorithm settings, and parameter values |
| 2 | The data is converted to an OO-LD document (ontology-mapped JSON) |
| 3 | The OO-LD is parsed into an RDF graph using both the base and extending schemas |
| 4 | The graph is validated against both SHACL shapes (base + extending) |
| 5 | Key facts and fitted parameters are extracted from the graph to confirm correctness |

To record a different calibration, edit `docs/example.input.json` and re-run all cells.


## Further reading

- [Schema patterns](../../../docs/schema-patterns.md): how inheritance works in this repository
- [Schema format reference](../../../docs/schema-format.md): how to write your own schema
- `simulation/step/PMDCo/` is the base schema this one extends
- `material-card/mechanical/PMDCo/` is the schema for the material card produced by this step
- See `workflow/PMDCo/docs/workflow_demo.ipynb` for a cross-domain demo showing how calibration connects to other steps